## ▶ What you'll see when you run this
- A **minimal MCP server** defined in code — exposing **one tool** (`get_price`) and **one resource** (`menu://today`) — and an **in-process client** that lists them and calls the tool, printing *`get_price(coffee) -> 4`*. No external server, no network.

**Time:** ~10 min · **Cost:** free (runs fully offline) · **Keys:** none required — uses the `mcp` Python SDK if installed, else a faithful no-deps stand-in that shows the SAME server structure.

# Week 13 · Notebook 4 — Build a Minimal MCP Server
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

MCP is the headline of this week and an A10 deliverable — so let's **see it in code**, not just prose. We build a tiny **MCP server** that exposes:
- **one tool** — `get_price(item)` (a function a client/Claude can call),
- **one resource** — `menu://today` (read-only data a client can fetch),

then connect a **client** to list and use them. This is the same shape as the GitHub / filesystem / Postgres servers you add to Claude Code with `claude mcp add ...`.

## 0. Install the official MCP SDK (graceful fallback)
We try the official `mcp` Python SDK. If it isn't installed or can't run here (e.g. Colab's event loop), we fall back to a **no-dependency stand-in** that mirrors the SAME server structure — so you always see how MCP is wired.

In [ ]:
!pip -q install mcp 2>/dev/null || echo 'mcp not installed — using the stand-in below'

In [ ]:
try:
    import mcp  # noqa: F401
    from mcp.server.fastmcp import FastMCP
    HAVE_MCP = True
    print('official mcp SDK available:', mcp.__version__ if hasattr(mcp,'__version__') else 'ok')
except Exception as e:
    HAVE_MCP = False
    print('mcp SDK not usable here -> using the stand-in. reason:', e)

## 1. Define the server with the official SDK
`FastMCP` turns decorated Python functions into MCP **tools** and **resources**. This is the *real* server you'd run as a process and register with `claude mcp add`. We define it whether or not the SDK is importable so you can read the structure either way.

In [ ]:
MENU = {'coffee': 4, 'tea': 3, 'cocoa': 5}

if HAVE_MCP:
    server = FastMCP('cafe')          # an MCP server named 'cafe'

    @server.tool()
    def get_price(item: str) -> int:
        """Look up a menu price in dollars."""   # tool: a callable action
        return MENU.get(item.lower(), 0)

    @server.resource('menu://today')
    def menu_today() -> str:
        """Today's menu as readable text."""      # resource: data to READ
        return '\n'.join(f'{k}: ${v}' for k, v in MENU.items())

    print('defined FastMCP server `cafe` with 1 tool + 1 resource')
    print('To run it as a real process:  server.run()  (stdio transport)')
else:
    print('SDK not importable here — see the annotated stand-in in section 2.')

## 2. A no-deps stand-in (SAME structure) + an in-process client
Identical shape — a registry of **tools** and **resources**, plus a tiny **client** that does what a real MCP client (Claude Code, the desktop app, your agent) does: **list** capabilities, then **call** a tool / **read** a resource over the protocol. This always runs.

In [ ]:
class MiniMCPServer:
    """A faithful (toy) MCP server: tools = callable actions, resources = readable data."""
    def __init__(self, name):
        self.name = name; self._tools = {}; self._resources = {}
    def tool(self, fn):
        self._tools[fn.__name__] = fn; return fn
    def resource(self, uri):
        def deco(fn): self._resources[uri] = fn; return fn
        return deco
    # ---- the 'protocol' surface a client talks to ----
    def list_tools(self):     return list(self._tools)
    def list_resources(self): return list(self._resources)
    def call_tool(self, name, **kwargs): return self._tools[name](**kwargs)
    def read_resource(self, uri):        return self._resources[uri]()

cafe = MiniMCPServer('cafe')

@cafe.tool
def get_price(item: str) -> int:
    """Look up a menu price in dollars."""     # TOOL
    return MENU.get(item.lower(), 0)

@cafe.resource('menu://today')
def menu_today() -> str:
    """Today's menu as text."""                 # RESOURCE
    return '\n'.join(f'{k}: ${v}' for k, v in MENU.items())

print('server:', cafe.name)

In [ ]:
# A client connects and uses the server over the (toy) protocol:
print('client -> list_tools     :', cafe.list_tools())
print('client -> list_resources :', cafe.list_resources())
print('client -> call get_price :', f"get_price(coffee) -> {cafe.call_tool('get_price', item='coffee')}")
print('client -> read resource  :')
print(cafe.read_resource('menu://today'))

## 3. How Claude / Claude Code connects to a real server
You don't usually call the server by hand — an **MCP client inside an AI app** does. With the real SDK you'd run the server as a process and register it:

```bash
# 1) run the server (stdio):   python cafe_server.py   # calls server.run()
# 2) register it with Claude Code:
claude mcp add cafe -- python /path/to/cafe_server.py
```
Now `get_price` shows up as a tool Claude can call and `menu://today` as a resource it can read — **no custom glue**. Write the server **once**; any MCP-aware client reuses it.

## 4. Security note — connecting MCP servers is an attack surface
An MCP server is **external code feeding your agent data and actions**, so the Week 13 guardrails apply directly:
- **Prompt injection via tool/resource results:** a server can return text that says *"ignore previous instructions…"*. Treat MCP results as **untrusted data**, not commands (scan them — see Notebook 3 / `agent_app.py`).
- **Over-broad permissions:** only connect servers you trust, and grant the **least privilege** needed — an `ALLOWED_TOOLS` allow-list still applies even when the tool arrives via MCP.
- **Human-in-the-loop** for consequential server tools (writes, payments).

### Takeaways
- An **MCP server** exposes **tools** (actions) + **resources** (data) [+ prompts]; an **MCP client** (in an AI app) lists and uses them.
- Write a tool **once** as a server; every MCP-aware client reuses it — *USB-C for AI tools*.
- MCP servers are an **attack surface**: untrusted results + least privilege + human-in-the-loop.

### Try it
1. Add a second tool to `cafe` (e.g. `is_open(hour)`); list + call it.
2. Add a `prompt` capability (a reusable template) — MCP's third primitive.
3. In one sentence: why does MCP beat hand-writing tool glue per app?

In [ ]:
# your MCP experiments here
